# Teacher Pile-KL vs. subsequent transfer, at a fixed student cell

**Question (2026-08-21).** Holding the student HP cell fixed, does the *teacher's* own
integrity damage (Pile-KL of the fine-tuned teacher vs its base, pp1 pool) predict how much
trait its numbers transfer? Monotone correlation is one possibility; a **sweet spot** is the
other — a teacher has to be pushed far enough to carry the trait into its numbers, but a
teacher pushed too far produces numbers that carry damage instead.

**Populations** (dog, `ex10-20_prefix_n100k` pools, every distillable teacher that got students):

| arm | teacher FT base → student base | teachers | student cells | ftseeds |
|---|---|---|---|---|
| **orig same-model** | `pythia-160m` → `pythia-160m` | ~102 | ep20 `lr1e-4_r8a32`, `lr4e-4_r32a4`, `lr4e-4_r16a4`, `lr3e-4_r16a8` | per cell summary |
| **ds1 same-model** | `pythia-160m-data-seed1` → `pythia-160m-data-seed1` | ~118 | std recipe `lr1e-4_r8a32` at ep 1 / 2 / 5 / 10 | 1 |
| orig → ds1 (cross-init, comparator) | `pythia-160m` → `pythia-160m-data-seed1` | ~102 | the 4 orig ep20 cells | per cell summary |

Teacher KL = `teachers/{animal}/{base}/{tag}/ftseed1/eval/pile_probe.json` (`kl_base_student`, pp1)
via the shared record cache; teacher trait strength = the teacher's own `validation` p(dog) from
its `preference_eval.json` (covariate). Student transfer = test $p(\text{dog})$ / the receiver's
test prior. Row unit throughout: one student run (teacher × cell × ftseed) unless stated.

**Selection caveat, stated up front.** Only teachers whose prefix-mode pool cleared the ≥10k
distillability gate have students, and on 160m that gate is almost exactly a teacher-KL cut
(≈2.7–2.8 nats, `collapse_teachers.ipynb`). So every teacher here has KL ≲ 2.8: the "too
damaged" side of a sweet spot can only be seen *inside* that range, and the ≥2.8 tail is
absent by construction, not because it transfers well or badly.

**See also.** `notebooks/teacher_quality_audit.ipynb` §2c answers the companion question with the
same-model *screen* as the predictor rather than teacher KL: over ~102 orig dog teachers the
same-model ep5 std transfer predicts cross transfer into every shared-init receiver (Spearman
+0.35 to +0.66 at the fixed `ep20_lr1e-4_r8a32` cell), while inside the canonical ten the
correlation vanishes through range restriction.


## 0. Setup

In [ ]:
from __future__ import annotations

import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats

from subliminal_learning.analysis import collapse_law as cl
from subliminal_learning.analysis import labels
from subliminal_learning.analysis import plots as aplots
from subliminal_learning.stats import core as astats
from subliminal_learning.analysis.loaders.frames import add_param_fields, load_base_prior, read_json
from subliminal_learning.analysis.loaders.probe_cache import load_teacher_probe_cache
from subliminal_learning.analysis.setup import bootstrap, set_plot_style

warnings.filterwarnings("ignore")
pd.set_option("display.width", 230)
set_plot_style(dpi=115, hide_spines=True)
ROOT, OUTPUT_ROOT = bootstrap()
ANIMAL, DATASET = "dog", "ex10-20_prefix_n100k"
KSTAR = cl.KSTAR_CALIBRATED

ARMS = {  # label -> (teacher base, student base, epochs filter, cells)
    "orig same-model": ("pythia-160m", "pythia-160m", 20,
                        ["ep20_lr1e-4_r8a32", "ep20_lr4e-4_r32a4", "ep20_lr4e-4_r16a4", "ep20_lr3e-4_r16a8"]),
    "ds1 same-model": ("pythia-160m-data-seed1", "pythia-160m-data-seed1", None,
                       ["ep1_lr1e-4_r8a32", "ep2_lr1e-4_r8a32", "ep5_lr1e-4_r8a32", "ep10_lr1e-4_r8a32"]),
    "orig -> ds1 (cross-init)": ("pythia-160m", "pythia-160m-data-seed1", 20,
                                 ["ep20_lr1e-4_r8a32", "ep20_lr4e-4_r32a4", "ep20_lr4e-4_r16a4", "ep20_lr3e-4_r16a8"]),
}
PRIOR_FILE = {"pythia-160m": "original", "pythia-160m-data-seed1": "ds1"}


def test_prior(base: str) -> float:
    return load_base_prior(OUTPUT_ROOT, PRIOR_FILE[base], ANIMAL)


def teacher_table(base: str) -> pd.DataFrame:
    # Pile-KL (pp1) of every fine-tuned teacher on this base, ftseed1, + its own trait strength
    t = load_teacher_probe_cache(OUTPUT_ROOT, f"teachers/{ANIMAL}/{base}", refresh=True)
    t = t[t.ftseed == 1][["param_tag", "kl", "delta_ce"]].rename(columns={"param_tag": "teacher", "kl": "teacher_kl"})
    val = {}
    for tag in t.teacher:
        f = OUTPUT_ROOT / "teachers" / ANIMAL / base / tag / "ftseed1" / "preference_eval.json"
        if f.exists():
            s = read_json(f)["sets"]
            val[tag] = (s.get("validation") or {}).get("p_target", np.nan)
    t["teacher_val_p"] = t.teacher.map(val)
    t = add_param_fields(t, "teacher", prefix="teacher")
    t["teacher_K"] = cl.K_of(t.teacher_lr, t.teacher_alpha, t.teacher_rank)
    return t

## 1. Load + inventory

Students via `cl.load_law_runs` over every teacher lineage directory present under the arm's
student tree (one row per student run, `test` = test p(dog)); teachers joined on the tag. The
inventory counts teachers with a KL record, runs, and the KL range actually covered.

`frac_dead` is the **legacy trait cut** (`collapse_law.DEAD_SEED` via `analysis.labels.
label_dead_trait_legacy`), not an integrity label: trait loss and integrity collapse are different
events. The operative label is the receiver's unigram-anchored Pile-KL cut, which these student
runs do not yet carry probes for.

In [ ]:
frames = []
for arm, (tbase, sbase, ep, cells) in ARMS.items():
    tree = OUTPUT_ROOT / "students" / ANIMAL / sbase / f"teacher-{tbase}"
    tags = sorted(d.name.removesuffix("_tseed1") for d in tree.glob("*_tseed1"))
    lineages = [(t, t, sbase) for t in tags]
    if ep is not None:
        _, r = cl.load_law_runs(OUTPUT_ROOT, animal=ANIMAL, numbers_base=tbase, dataset=DATASET, epochs=ep, lineages=lineages)
    else:
        parts = [cl.load_law_runs(OUTPUT_ROOT, animal=ANIMAL, numbers_base=tbase, dataset=DATASET, epochs=e, lineages=lineages)[1]
                 for e in (1, 2, 5, 10)]
        r = pd.concat(parts, ignore_index=True)
    r = r[r.student_tag.isin(cells)].rename(columns={"family": "teacher"})
    r = r.merge(teacher_table(tbase), on="teacher", how="left")
    r["arm"], r["prior"] = arm, test_prior(sbase)
    r["transfer"] = r.test / r.prior
    r["log_transfer"] = np.log10(r.transfer.clip(lower=1e-3))
    # legacy trait cut — caveat: trait loss != integrity collapse; operative label = anchored KL (analysis.labels)
    # TODO(anchored): wire Pile-KL when probes land for these student runs
    r["dead"] = labels.label_dead_trait_legacy(r).astype(float)
    frames.append(r)
runs = pd.concat(frames, ignore_index=True)

inv = runs.groupby("arm").agg(teachers=("teacher", "nunique"), teachers_with_kl=("teacher_kl", lambda v: runs.loc[v.index].dropna(subset=["teacher_kl"]).teacher.nunique()),
                              cells=("student_tag", "nunique"), runs=("test", "size"), ftseeds=("seed", "nunique"),
                              kl_min=("teacher_kl", "min"), kl_med=("teacher_kl", "median"), kl_max=("teacher_kl", "max"),
                              frac_dead=("dead", "mean"))
display(inv.round(3))
runs = runs.dropna(subset=["teacher_kl"]).copy()

## 2. The raw picture — teacher KL vs transfer, per cell

One panel per (arm, cell). x = teacher Pile-KL (nats), y = student transfer (log). Points are
runs; the thick line is the median of transfer in 0.25-nat teacher-KL bins (≥ 4 runs per bin);
the horizontal rule is the receiver prior. If there is a sweet spot the binned median rises
then falls inside the observed KL range.

In [ ]:
def binned(df, x="teacher_kl", y="transfer", w=0.25, min_n=4):
    b = (df[x] // w) * w + w / 2
    g = df.groupby(b)[y]
    m = g.median(); n = g.size()
    return m[n >= min_n], n[n >= min_n]


for arm, (_, _, _, cells) in ARMS.items():
    sub = runs[runs.arm == arm]
    fig, axes = plt.subplots(1, len(cells), figsize=(4.2 * len(cells), 4.0), sharey=True)
    for ax, cell in zip(axes, cells):
        s = sub[sub.student_tag == cell]
        ax.scatter(s.teacher_kl, s.transfer.clip(lower=1e-3), s=14, alpha=0.45, color="#1c7ed6")
        m, n = binned(s)
        ax.plot(m.index, m.values, "-o", color="#d9480f", lw=2.2, ms=5, label="median per 0.25-nat bin")
        ax.axhline(1, color="0.3", lw=1)
        rho = stats.spearmanr(s.teacher_kl, s.transfer).correlation if len(s) > 5 else np.nan
        ax.set(yscale="log", title=f"{cell}  (n={len(s)}, rho={rho:+.2f})", xlabel="teacher Pile-KL (nats)")
    axes[0].set_ylabel("transfer = test p(dog) / prior (log)")
    axes[0].legend(fontsize=7, loc="lower left")
    fig.suptitle(f"{arm}: teacher integrity damage vs student transfer, per student cell; row unit = one run", y=1.03)
    plt.tight_layout(); plt.show()

## 3. Monotone or humped? — per-cell tests

For each (arm, cell): Spearman ρ(teacher KL, transfer); a quadratic fit
$\log_{10}\text{transfer} = a + b\,\mathrm{KL} + c\,\mathrm{KL}^2$ (cluster-robust on teacher) with
the sign/CI of $c$ and the implied vertex $\mathrm{KL}^\ast=-b/2c$ when it is a maximum inside
the observed range; and the KL of the top-decile teachers (by transfer) vs the rest. A hump
shows as $c<0$ with an interior vertex and top-decile KL sitting mid-range; a monotone relation
shows as a one-signed ρ and a vertex outside the range.

In [ ]:
rows = []
for arm, (_, _, _, cells) in ARMS.items():
    for cell in cells:
        s = runs[(runs.arm == arm) & (runs.student_tag == cell)].copy()
        if len(s) < 10:
            continue
        s["kl2"] = s.teacher_kl ** 2
        f = astats.cluster_ols(s, "log_transfer ~ teacher_kl + kl2", "teacher")
        b, c = f.params["teacher_kl"], f.params["kl2"]
        lo, hi = s.teacher_kl.min(), s.teacher_kl.max()
        vertex = -b / (2 * c) if c != 0 else np.nan
        interior_max = (c < 0) and (lo < vertex < hi)
        top = s.sort_values("transfer", ascending=False).head(max(3, len(s) // 10))
        rows.append({"arm": arm, "cell": cell, "n": len(s), "n_teachers": s.teacher.nunique(),
                     "spearman": stats.spearmanr(s.teacher_kl, s.transfer).correlation,
                     "spearman_p": stats.spearmanr(s.teacher_kl, s.transfer).pvalue,
                     "quad_c": c, "quad_c_p": f.pvalues["kl2"], "vertex_KL": vertex, "interior_max": interior_max,
                     "KL_range": (round(lo, 2), round(hi, 2)),
                     "top_decile_KL_med": top.teacher_kl.median(), "rest_KL_med": s.drop(top.index).teacher_kl.median(),
                     "top_decile_transfer_med": top.transfer.median()})
tests = pd.DataFrame(rows).set_index(["arm", "cell"])
display(tests.round(3))

## 4. Pooled across cells, with the confounds named

Teacher KL is itself a function of the teacher's recipe (its own dose $K_T$ and epochs), and so is
the trait strength the teacher reached (`validation` p). Three pooled views per arm (cell fixed
effects so the student HP is held constant):

1. transfer vs teacher KL (binned medians over all cells, 0.25-nat bins);
2. the same with the teacher's **own trait strength** on x — does KL add anything once you know
   how strongly the teacher itself holds the trait?
3. a cell-FE regression of $\log_{10}$ transfer on KL, KL², teacher validation-p and
   $\log K_T$, cluster-robust on teacher.

The second row of the figure repeats view (1) with the teacher's **Pile ΔCE** (pp1, nats, CE-primary
convention) on x instead of KL, and **transfer on a linear y-axis** so the plateau height is read
directly rather than in decades. Same 0.25-nat bins, same cells pooled, same binned medians.


In [ ]:
def trait_strength(val_p):
    """Teacher trait strength on a log-odds-ish scale: -log10(1 - validation p), clipped."""
    return -np.log10(1 - val_p.clip(upper=1 - 1e-6))


fig, axes = plt.subplots(2, 3, figsize=(17, 9.4))
colors = aplots.palette(list(ARMS))
# row 0: transfer on log y, teacher Pile-KL on x (as before)
# row 1: transfer on LINEAR y, teacher Pile delta-CE (nats) on x
for arm in ARMS:
    s = runs[runs.arm == arm]
    tv = trait_strength(s.teacher_val_p)
    m, n = binned(s)
    axes[0, 0].plot(m.index, m.values, "-o", color=colors[arm], lw=2.2, ms=5, label=f"{arm} (n={len(s)})")
    mv, _ = binned(s.assign(tv=tv), x="tv", w=0.25)
    axes[0, 1].plot(mv.index, mv.values, "-o", color=colors[arm], lw=2.2, ms=5, label=arm)
    axes[0, 2].scatter(s.teacher_kl, tv, s=10, alpha=0.4, color=colors[arm], label=arm)
    mc, _ = binned(s, x="delta_ce")
    axes[1, 0].plot(mc.index, mc.values, "-o", color=colors[arm], lw=2.2, ms=5, label=f"{arm} (n={s.delta_ce.notna().sum()})")
    axes[1, 1].plot(mv.index, mv.values, "-o", color=colors[arm], lw=2.2, ms=5, label=arm)
    axes[1, 2].scatter(s.delta_ce, tv, s=10, alpha=0.4, color=colors[arm], label=arm)
for ax in axes[:, :2].ravel():
    ax.axhline(1, color="0.3", lw=1)
for ax in axes[0, :2]:
    ax.set_yscale("log")
axes[0, 0].set(xlabel="teacher Pile-KL (nats)", ylabel="median transfer (log)", title="transfer vs teacher KL (all cells pooled, binned medians)")
axes[0, 1].set(xlabel="teacher trait strength: -log10(1 - validation p)", title="transfer vs teacher's own trait strength (log y)")
axes[0, 2].set(xlabel="teacher Pile-KL", ylabel="-log10(1 - validation p)", title="the two teacher covariates against each other")
axes[1, 0].set(xlabel="teacher Pile ΔCE (nats)", ylabel="median transfer (linear)", title="transfer vs teacher ΔCE (all cells pooled, 0.25-nat bins, linear y)")
axes[1, 1].set(xlabel="teacher trait strength: -log10(1 - validation p)", title="transfer vs teacher's own trait strength (linear y)")
axes[1, 2].set(xlabel="teacher Pile ΔCE (nats)", ylabel="-log10(1 - validation p)", title="teacher ΔCE vs trait strength")
aplots.right_legend(axes[0, 2], fontsize=8)
aplots.right_legend(axes[1, 2], fontsize=8)
plt.tight_layout(); plt.show()

for arm in ARMS:
    s = runs[runs.arm == arm].copy()
    s["kl2"] = s.teacher_kl ** 2; s["tv"] = trait_strength(s.teacher_val_p); s["logK"] = np.log10(s.teacher_K)
    f1 = astats.cluster_ols(s, "log_transfer ~ C(student_tag) + teacher_kl + kl2", "teacher")
    f2 = astats.cluster_ols(s, "log_transfer ~ C(student_tag) + teacher_kl + kl2 + tv + logK", "teacher")
    c, b = f1.params["kl2"], f1.params["teacher_kl"]
    print(f"{arm}: n={len(s)} | KL-only (cell FE): b={b:+.3f} c={c:+.3f} (p={f1.pvalues['kl2']:.3g}), vertex KL={-b/(2*c) if c else np.nan:.2f}, R2={f1.rsquared:.3f}"
          f" | + teacher val-p + log K_T: KL p={f2.pvalues['teacher_kl']:.3g}, KL^2 p={f2.pvalues['kl2']:.3g}, val-p coef={f2.params['tv']:+.3f} (p={f2.pvalues['tv']:.3g}), logK coef={f2.params['logK']:+.3f} (p={f2.pvalues['logK']:.3g}), R2={f2.rsquared:.3f}")

### 4b. Does KL add ordering where validation-p is saturated?

The objection: "higher validation-score teachers are better" was already known, but
validation p saturates at ~1 almost immediately, so it cannot *order* the good teachers. Does
teacher KL order them? Restrict to teachers whose own validation p(dog) ≥ a threshold (0.9, 0.99,
0.999) and, within that subset, report Spearman(KL, transfer) and Spearman(val-p, transfer) per
arm (cell fixed: computed per cell, then the median over cells), plus the spread of transfer the
subset still has. If KL keeps a large ρ where val-p's ρ collapses, KL is the usable ordering
variable among saturated teachers. Row unit: one run; ρ per cell, median across the arm's cells.

In [ ]:
rows = []
for arm, (_, _, _, cells) in ARMS.items():
    for thr in (0.0, 0.9, 0.99, 0.999):
        sub = runs[(runs.arm == arm) & (runs.teacher_val_p >= thr)]
        rk, rv, ns = [], [], []
        for cell in cells:
            s = sub[sub.student_tag == cell]
            if s.teacher.nunique() < 8:
                continue
            rk.append(stats.spearmanr(s.teacher_kl, s.transfer).correlation)
            rv.append(stats.spearmanr(s.teacher_val_p, s.transfer).correlation)
            ns.append(s.teacher.nunique())
        if not ns:
            continue
        rows.append({"arm": arm, "val_p >=": thr, "teachers": int(np.median(ns)), "rho_KL (median over cells)": np.median(rk),
                     "rho_valp (median over cells)": np.median(rv),
                     "val_p range": (round(sub.teacher_val_p.min(), 4), round(sub.teacher_val_p.max(), 4)),
                     "KL range": (round(sub.teacher_kl.min(), 2), round(sub.teacher_kl.max(), 2)),
                     "transfer p10/p90": (round(sub.transfer.quantile(.1), 1), round(sub.transfer.quantile(.9), 1))})
sat = pd.DataFrame(rows).set_index(["arm", "val_p >="])
display(sat.round(3))

# the same thing visually for the same-model arms: among val-p >= 0.99 teachers, transfer vs KL
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), sharey=True)
for ax, arm in zip(axes, ("orig same-model", "ds1 same-model")):
    sub = runs[(runs.arm == arm) & (runs.teacher_val_p >= 0.99)]
    for cell, c in zip(ARMS[arm][3], aplots.palette(ARMS[arm][3]).values()):
        s = sub[sub.student_tag == cell]
        ax.scatter(s.teacher_kl, s.transfer, s=16, alpha=0.6, color=c, label=f"{cell} (n={len(s)})")
        m, _ = binned(s)
        ax.plot(m.index, m.values, "-", color=c, lw=2)
    ax.set(xlabel="teacher Pile-KL (nats)", yscale="log", title=f"{arm}: teachers with validation p >= 0.99 only")
    ax.legend(fontsize=7)
axes[0].set_ylabel("transfer = test p / prior (log)")
plt.tight_layout(); plt.show()

## 5. Reading

Restated from the executed cells (see the tables above for the numbers behind each line).

In [ ]:
for arm in ARMS:
    t = tests.loc[arm]
    print(f"{arm}:")
    for cell, r in t.iterrows():
        # a negative quadratic term with a vertex in the upper part of the range and no drop in the
        # top bins is SATURATION, not a hump: check the binned medians directly
        s_ = runs[(runs.arm == arm) & (runs.student_tag == cell)]
        m, _ = binned(s_)
        drop = (m.iloc[-2:].mean() / m.max()) if len(m) >= 3 else np.nan
        if r.interior_max and drop < 0.7:
            shape = "HUMP (interior max at KL~%.2f; top bins at %.0f%% of peak)" % (r.vertex_KL, 100 * drop)
        elif r.spearman > 0.5:
            shape = "monotone, SATURATING (top bins at %.0f%% of peak median)" % (100 * drop)
        else:
            shape = "monotone +" if r.spearman > 0.15 else "monotone -" if r.spearman < -0.15 else "flat/weak"
        print(f"   {cell:20s} n={int(r.n):4d} rho={r.spearman:+.2f} (p={r.spearman_p:.2g})  quad c={r.quad_c:+.2f} (p={r.quad_c_p:.2g})  -> {shape}; "
              f"top-decile teachers KL {r.top_decile_KL_med:.2f} vs rest {r.rest_KL_med:.2f}")
print("\nCaveats: orig-tree teachers are all on the distillable side of the ~2.7-nat gate (the far side of any sweet spot is censored);\n"
      "ds1-tree teachers reach KL ~4.9 and STILL saturate rather than decline. S4 shows KL has no effect once the teacher\x27s own\n"
      "trait strength (validation p) and recipe dose log K_T are in the model: KL is a proxy for how far the teacher was pushed.")